# VLM-DENTAL — Stage 1: Supervised Fine-Tuning (SFT)

This notebook trains **Qwen3.5-9B** using native **BF16 LoRA** ($r=32, \alpha=64$) on verified dental clinical traces.

### Core Architectural Invariants:
- **Hardware Optimization**: Native support for **Google Cloud TPU v5e-8** (8-way FSDPv2) and multi-GPU (DDP / Accelerate).
- **Strict Track Segregation**:
  - **Track A (`with_tools`)**: Multi-turn agent trained on real workstation tool-use traces (8 tools).
  - **Track B (`no_tools`)**: Single-turn direct radiologist trained on tool-free Chain-of-Thought (CoT) traces.
- **3D MRoPE & Bucketing**: Right-padding only (`padding_side = "right"`) with static discrete sequence length buckets (`[4096, 6144, 8192, 10240]` for Track A; `[1536, 2048, 2560, 3072]` for Track B) to eliminate XLA dynamic recompilations.
- **Conversational Assistant-Only Loss Masking**: Loss is computed strictly on assistant clinical reasoning, tool call JSON, and `<|im_end|>`. System prompts, user queries, and tool returns are masked with `-100`.
- **Hugging Face Hub Checkpoint Sync**: Checkpoints (~760 MB LoRA + optimizer) are pushed every 25 steps to survive Kaggle 9-hour session limits and enable seamless multi-account resume.

## 1. Hardware Auto-Detection (TPU v5e-8 vs GPU)

In [ ]:
import os
import sys
import torch

IS_TPU = False
DEVICE_STR = "cpu"

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    IS_TPU = True
    DEVICE_STR = f"TPU ({xm.xla_device_hw(device)}) - {device}"
    print(f"[HARDWARE] Detected Cloud TPU: {DEVICE_STR}")
except Exception:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        DEVICE_STR = f"GPU ({gpu_count}x {gpu_name})"
        print(f"[HARDWARE] Detected CUDA: {DEVICE_STR}")
    else:
        print("[HARDWARE] Running on CPU (Testing only).")

print(f"PyTorch Version: {torch.__version__}")

## 2. Environment Setup & Dependency Installation

In [ ]:
# Clone or pull latest VLM-DENTAL codebase
if not os.path.exists("VLM-DENTAL"):
    !git clone https://github.com/rezaxr14/VLM-DENTAL.git
    %cd VLM-DENTAL
else:
    %cd VLM-DENTAL
    !git pull

# Install project and dependencies
!pip install -q -e .
!pip install -q peft trl datasets accelerate huggingface_hub ultralytics python-dotenv
!pip install -q qwen-vl-utils

## 3. Hugging Face Authentication & Trace Verification

In [ ]:
from huggingface_hub import login

# Load HF Token from environment or prompt
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[AUTH] Logged into Hugging Face via HF_TOKEN.")
else:
    print("[AUTH] Please log into Hugging Face:")
    login()

# Sync verified canonical traces from HF Hub if not found locally
traces_dir = "data/traces"
os.makedirs(traces_dir, exist_ok=True)

target_trace_files = [
    "train_cot_traces.jsonl",
    "train_cot_traces_no_tools.jsonl",
    "train_cot_traces_healthy_tufts.jsonl"
]

missing = [f for f in target_trace_files if not os.path.exists(os.path.join(traces_dir, f))]
if missing:
    print(f"[SYNC] Downloading missing verified trace files: {missing}...")
    !python scripts/sync_traces_hf.py --download
else:
    print("[SYNC] All verified canonical trace files are present locally.")

## 4. Training Configuration & Track Selection

In [ ]:
# =========================================================================
# STAGE 1 SFT EXECUTION PARAMETERS
# =========================================================================
# Choose track: "with_tools" (multi-turn agent) or "no_tools" (direct radiologist)
TRACK = "with_tools"

# Precision: "bf16" (recommended on TPU v5e-8 and Ampere+ GPUs), "fp16", or "qlora"
PRECISION = "bf16"

# Hugging Face Checkpoint Repository for multi-account Kaggle continuity
HF_REPO = "Reza-Nadimi/vlm-dental-checkpoints"

# Push checkpoint every N steps to avoid losing compute on 9h Kaggle timeout
PUSH_EVERY_STEPS = 25

# Set RESUME = True if continuing from a previously uploaded HF checkpoint
RESUME = False

print(f"[CONFIG] Track: {TRACK}")
print(f"[CONFIG] Precision: {PRECISION}")
print(f"[CONFIG] HF Checkpoint Hub: {HF_REPO}")
print(f"[CONFIG] Auto-upload every: {PUSH_EVERY_STEPS} steps")

## 5. Launch SFT Training Pipeline

In [ ]:
cmd = [
    "python", "scripts/train_sft.py",
    "--track", TRACK,
    "--precision", PRECISION,
    "--hf-repo", HF_REPO,
    "--push-every-steps", str(PUSH_EVERY_STEPS),
    "--epochs", "3",
    "--learning-rate", "2e-5",
    "--lora-r", "32",
    "--lora-alpha", "64"
]

if RESUME:
    cmd.extend(["--resume-hf", HF_REPO])

cmd_str = " ".join(cmd)
print(f"[EXECUTE] Running: {cmd_str}")
!{cmd_str}

## 6. Training Loss & Convergence Visualizer

In [ ]:
import json
import matplotlib.pyplot as plt

output_tag = "qwen3_5_9b_sft_tools" if TRACK == "with_tools" else "qwen3_5_9b_sft_no_tools"
log_file = f"data/models/{output_tag}/training_loss.jsonl"

if os.path.exists(log_file):
    steps, losses = [], []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                steps.append(record.get("step", len(steps)))
                losses.append(record.get("loss", 0.0))
                
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label="SFT Assistant Loss", color="#1f77b4", lw=2)
    plt.title(f"VLM-DENTAL Stage 1 SFT Loss Curve — Track: {TRACK}")
    plt.xlabel("Training Steps")
    plt.ylabel("Conversational Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    print(f"[RESULTS] Initial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}")
else:
    print(f"[INFO] Log file {log_file} not yet generated. Run training above to view metrics.")